# Solution A3 · Standard errors of the coefficients: closed formula versus bootstrap

**Type**: Implement from scratch

## Objective

Compute the uncertainty of the OLS coefficients in two ways, with the analytic formula and with the pairs bootstrap, and
show that the two results coincide only if the errors are homoscedastic.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

TRUE_BETA = np.array([1.0, 2.0, -1.0, 0.5])


def generate(heteroscedastic, seed=1, n=150):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, 3))
    sigma = 2.0 * (1.5 * np.abs(X[:, 0]) + 0.1) if heteroscedastic else 2.0
    y = TRUE_BETA[0] + X @ TRUE_BETA[1:] + rng.normal(size=n) * sigma
    return X, y


X_homo, y_homo = generate(False)
X_het, y_het = generate(True)

## Commented solution

In [ ]:
def standard_errors_formula(X, y):
    A = np.column_stack([np.ones(len(X)), X])
    n, k = A.shape
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    residuals = y - A @ beta
    sigma2 = residuals @ residuals / (n - k)          # unbiased estimate of the error variance
    return np.sqrt(np.diag(sigma2 * np.linalg.inv(A.T @ A)))


def standard_errors_bootstrap(X, y, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    A = np.column_stack([np.ones(len(X)), X])
    coef_boot = np.empty((B, A.shape[1]))
    for b in range(B):
        idx = rng.integers(0, len(y), len(y))         # resample the ROWS, (x, y) pairs together
        coef_boot[b] = np.linalg.lstsq(A[idx], y[idx], rcond=None)[0]
    return coef_boot.std(axis=0, ddof=1), coef_boot

In [ ]:
se_formula_homo = standard_errors_formula(X_homo, y_homo)
se_boot_homo, coef_boot_homo = standard_errors_bootstrap(X_homo, y_homo)
ratio_homo = se_boot_homo / se_formula_homo
print("homoscedastic   - bootstrap/formula ratio:", ratio_homo.round(2))

In [ ]:
se_formula_het = standard_errors_formula(X_het, y_het)
se_boot_het, _ = standard_errors_bootstrap(X_het, y_het)
ratio_het = se_boot_het / se_formula_het
underestimates_x1 = bool(ratio_het[1] > 1.3)
print("heteroscedastic - bootstrap/formula ratio:", ratio_het.round(2))

In [ ]:
ci = np.percentile(coef_boot_homo, [2.5, 97.5], axis=0)
ci_width = ci[1] - ci[0]
print("95% intervals:")
print(pd.DataFrame(ci.T, columns=["2.5%", "97.5%"], index=["intercept", "x1", "x2", "x3"]).assign(true=TRUE_BETA).round(2))

Cross-check with the reference statistics library in Python: the analytic standard errors must
coincide with those of `statsmodels` (which uses the same formula).

In [ ]:
import statsmodels.api as sm
ols = sm.OLS(y_homo, sm.add_constant(X_homo)).fit()
print("statsmodels:", ols.bse.round(4))
print("yours      :", se_formula_homo.round(4))

## Check

In [ ]:
with verify():
    A = np.column_stack([np.ones(len(X_homo)), X_homo])
    e = y_homo - A @ np.linalg.lstsq(A, y_homo, rcond=None)[0]
    expected = np.sqrt(np.diag(e @ e / (len(y_homo) - 4) * np.linalg.inv(A.T @ A)))
    check(close(se_formula_homo, expected), "the analytic standard errors are correct",
          "standard_errors_formula does not match sqrt(diag(sigma2 * inv(A^T A))): check the degrees of freedom n-p-1")
    check(coef_boot_homo.shape == (2000, 4), "the bootstrap returns B x 4 estimates", "coef_boot must have shape (B, 4)")
    check(np.all((ratio_homo > 0.85) & (ratio_homo < 1.2)),
          "with homoscedastic errors formula and bootstrap agree (ratios between 0.85 and 1.2)",
          "on the homoscedastic dataset the two methods should give similar standard errors")
    check(bool(underestimates_x1) and ratio_het[1] > 1.3,
          "with heteroscedastic errors the formula underestimates the uncertainty of the first feature",
          "on the heteroscedastic dataset the ratio for the first feature should exceed 1.3")
    check(close(ci_width, 3.92 * se_boot_homo, rtol=0.15),
          "the 95% intervals are about 3.92 standard errors wide",
          "ci_width should be ci[1] - ci[0], that is about 2 * 1.96 * se")
done()

## What you learned

- The formula `σ̂² (AᵀA)⁻¹` holds **only** if the errors have constant variance and are independent. When the
  variance grows with the feature, the extreme points (high leverage) are also the ones with the largest error, and the formula
  treats them as if they were normal noise: the uncertainty on the slope is underestimated (here by about 40%).
- The pairs bootstrap assumes nothing about the shape of the variance: it is an honest estimate even when the
  classical assumptions fail. It costs B fits instead of one, but on small data it is irrelevant.
- A 95% confidence interval is about 4 standard errors wide: if the standard error is wrong, so are the
  p-values and the intervals.

## Common pitfall

The bootstrap must be done on the **rows** (x, y pairs). Resampling only the residuals is a different procedure (residual
bootstrap) that presupposes homoscedasticity itself, the very assumption you want to test here.

## Interview question

<details><summary>In a regression the p-values of the coefficients look great but the residual plot shows a funnel. What do you do?</summary>

The funnel indicates heteroscedasticity: the coefficient estimates remain unbiased, but the classical standard errors are
wrong, and so are the p-values and intervals. I would recompute the uncertainty with robust standard errors (HC) or with the
pairs bootstrap, and I would check whether a transformation of the target (for example the logarithm) stabilizes the variance.

</details>